# 07 — True daily ranker lab

Purpose: executable true LightGBM LambdaRank OOS experiment. This notebook fits a daily cross-sectional ranker on the train period only, predicts out-of-sample scores on the test period, and evaluates against raw 10D forward returns. The rank target is a processed training-only daily percentile; economic evaluation always uses raw unprocessed 10D returns. This is a genuine ranker — not a regression-score rank transform.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init
from src.research.daily_ranker import prepare_ranker_frame
from src.research.daily_ranker_model import fit_lgbm_daily_ranker, predict_lgbm_daily_ranker
from src.research.notebook_lab_contracts import ResearchSessionConfig, CANONICAL_10D_RETURN_EXPR
from src.research.notebook_research_api import sanitize_factor_name
from src.research.notebook_experiment_api import run_10d_experiment
from src.research.ten_day_model_gates import summarize_report_gates

cfg_path = ROOT / "data" / "session_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    MARKET, SYMBOLS, BENCHMARK = cfg["market"], cfg["symbols"], cfg["benchmark"]
    TRAIN_START, TRAIN_END = cfg["train_start"], cfg["train_end"]
    TEST_START, TEST_END = cfg["test_start"], cfg["test_end"]
else:
    cfg = None
    MARKET = "us"
    SYMBOLS = ["AAPL", "NVDA", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "AVGO", "COST", "NFLX"]
    BENCHMARK = "QQQ"
    TRAIN_START, TRAIN_END = "2021-01-01", "2024-12-31"
    TEST_START, TEST_END = "2025-01-01", "2026-06-18"

# Cross-sectional ranking requires at least 2 symbols for meaningful rank comparison
assert len(SYMBOLS) >= 2, f"Cross-sectional ranking requires at least 2 symbols, got {len(SYMBOLS)}"

# Derive topk from config (default 3), validate it is a positive integer,
# then clamp to at most len(SYMBOLS)-1 so the portfolio never owns the whole universe
requested_topk = cfg.get("topk", 3) if cfg else 3
assert isinstance(requested_topk, int) and requested_topk > 0, (
    f"topk must be a positive integer, got {requested_topk}"
)
TOPK = min(requested_topk, len(SYMBOLS) - 1)

config = ResearchSessionConfig(
    market=MARKET,
    symbols=SYMBOLS,
    benchmark=BENCHMARK,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
    topk=TOPK,
    model_type="lgbm_lambdarank",
    experiment_id=f"{MARKET}_true_daily_ranker_lab",
    return_expression=CANONICAL_10D_RETURN_EXPR,
)

# Contract: model_type must identify the ranker, topk must not cover the whole universe
assert config.model_type == "lgbm_lambdarank", (
    f"model_type must be lgbm_lambdarank for genuine LambdaRank training, "
    f"got {config.model_type}"
)
assert 1 <= config.topk < len(config.symbols), (
    f"topk ({config.topk}) must be >= 1 and < n_symbols ({len(config.symbols)}) "
    f"— otherwise the portfolio holds the entire universe and "
    f"economic evaluation (return/turnover/drawdown) is degenerate"
)

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET, provider_uri_default=str(ROOT / "data" / "watchlist")))
from qlib.data import D

print(config)

In [ ]:
# ── Load features across FULL period, raw returns across FULL period ──
feature_exprs = [
    "$close/Ref($close,5)-1",
    "$close/Ref($close,10)-1",
    "$close/Ref($close,20)-1",
    "Std($ret,10)",
    "$volume/Ref($volume,10)-1",
]

features_all = D.features(SYMBOLS, feature_exprs, start_time=TRAIN_START, end_time=TEST_END)
raw_all = D.features(SYMBOLS, [config.return_expression], start_time=TRAIN_START, end_time=TEST_END)

for frame in (features_all, raw_all):
    if frame.index.names == ["instrument", "datetime"]:
        frame.index = frame.index.swaplevel()
        frame.sort_index(inplace=True)

features_all = features_all.fillna(0.0).replace([np.inf, -np.inf], 0.0)
features_all.columns = [sanitize_factor_name(expr) for expr in feature_exprs]

raw_all.columns = ["return"]
raw_all.attrs["provenance"] = "raw_forward_return"
raw_all.attrs["horizon"] = 10
raw_all.attrs["expression"] = config.return_expression

# ── Split by date ──
dates = features_all.index.get_level_values("datetime")
train_mask = (dates >= pd.Timestamp(TRAIN_START)) & (dates <= pd.Timestamp(TRAIN_END))
test_mask = (dates >= pd.Timestamp(TEST_START)) & (dates <= pd.Timestamp(TEST_END))

features_train = features_all.loc[train_mask].copy()
features_test = features_all.loc[test_mask].copy()
returns_train = raw_all.loc[train_mask].copy()
returns_test = raw_all.loc[test_mask].copy()
returns_test.attrs.update(raw_all.attrs)

# ── Purge OOS label leakage: the final holding_days training dates have
# Ref($close, -10) labels that peek into the test period. Purge those
# dates from training to close the leakage before prepare_ranker_frame. ──
all_trading_dates = sorted(features_all.index.get_level_values("datetime").unique())
common_train_dates = sorted(
    set(features_train.index.get_level_values("datetime").unique())
    & set(returns_train.index.get_level_values("datetime").unique())
)
assert len(common_train_dates) > config.holding_days, (
    f"Need > {config.holding_days} training trading dates for embargo purge, "
    f"got {len(common_train_dates)}"
)

purge_dates = set(common_train_dates[-config.holding_days:])
purge_boundary = common_train_dates[-config.holding_days - 1]

keep_feat = ~features_train.index.get_level_values("datetime").isin(purge_dates)
features_train = features_train.loc[keep_feat].copy()
keep_ret = ~returns_train.index.get_level_values("datetime").isin(purge_dates)
returns_train = returns_train.loc[keep_ret].copy()
returns_train.attrs.update(raw_all.attrs)

print(f"Leakage embargo: purged {len(purge_dates)} trading dates "
      f"({min(purge_dates).date()} → {max(purge_dates).date()})")
print(f"Last retained training date: {purge_boundary.date()}")

# Contract: latest label-training date precedes earliest test date by ≥ holding_days trading days
first_test_date = sorted(returns_test.index.get_level_values("datetime").unique())[0]
purge_boundary_pos = all_trading_dates.index(purge_boundary)
first_test_pos = all_trading_dates.index(first_test_date)
trading_date_gap = first_test_pos - purge_boundary_pos
assert trading_date_gap >= config.holding_days, (
    f"Embargo gap insufficient: {trading_date_gap} trading days between "
    f"latest training date ({purge_boundary.date()}) and first test date "
    f"({first_test_date.date()}), need ≥ {config.holding_days}"
)
print(f"Embargo gap: {trading_date_gap} trading days ≥ {config.holding_days} ✓")

# Contract: purged dates must not enter X_rank / y_rank
assert purge_boundary not in purge_dates, (
    f"purge boundary {purge_boundary.date()} must not be in purge set"
)

print(f"train features: {features_train.shape} | test features: {features_test.shape}")
print(f"train returns: {returns_train.shape} | test returns: {returns_test.shape}")

In [ ]:
# ── Build training-only rank target & groups (NEVER scored as economic returns) ──
X_rank, y_rank, groups = prepare_ranker_frame(features_train, returns_train)

# Direct assertion: X_rank and y_rank datetime levels must exclude purged dates
x_dates = set(X_rank.index.get_level_values("datetime"))
y_dates = set(y_rank.index.get_level_values("datetime"))
assert x_dates.isdisjoint(purge_dates), (
    f"X_rank contains {len(x_dates & purge_dates)} purged trading dates"
)
assert y_dates.isdisjoint(purge_dates), (
    f"y_rank contains {len(y_dates & purge_dates)} purged trading dates"
)

print(f"ranker features: {X_rank.shape}")
print(f"rank target: {y_rank.shape}")
print(f"groups: {len(groups)} dates, first five sizes={groups[:5]}")
print(f"processed rank target provenance: {y_rank.attrs.get('provenance')}")
print(y_rank.describe())

In [ ]:
# ── Fit true LightGBM LambdaRank on train period only ──
ranker_result = fit_lgbm_daily_ranker(
    X_rank,
    y_rank,
    groups,
    n_gain_bins=5,
    num_boost_round=200,
)

print(f"model type: {type(ranker_result.model).__module__}.{type(ranker_result.model).__qualname__}")
print(f"feature_names: {ranker_result.feature_names}")
print(f"n_gain_bins: {ranker_result.n_gain_bins}")
print(f"groups (first 5): {ranker_result.groups[:5]}")
assert ranker_result.n_gain_bins == 5
assert len(ranker_result.feature_names) == len(feature_exprs)

In [ ]:
# ── Predict OOS scores ONLY on the test period ──
ranker_scores = predict_lgbm_daily_ranker(ranker_result, features_test)

print(f"OOS predictions: {ranker_scores.shape}")
print(f"provenance: {ranker_scores.attrs.get('provenance')}")
print(f"model_type: {ranker_scores.attrs.get('model_type')}")
print(ranker_scores.head())
print(ranker_scores.describe())

# Contract: predictions must NOT overlap with train dates
train_dates_used = set(features_train.index.get_level_values("datetime"))
test_dates_used = set(ranker_scores.index.get_level_values("datetime"))
assert train_dates_used.isdisjoint(test_dates_used), "train/test date leakage detected"
print(f"train dates: {len(train_dates_used)} | test dates: {len(test_dates_used)} | overlap: 0 ✓")

In [ ]:
# ── Build historical momentum baseline: $close/Ref($close,10)-1 ──
baseline_raw = D.features(SYMBOLS, ["$close/Ref($close,10)-1"], start_time=TEST_START, end_time=TEST_END)
if baseline_raw.index.names == ["instrument", "datetime"]:
    baseline_raw = baseline_raw.swaplevel().sort_index()
baseline = baseline_raw.copy()
baseline.columns = ["score"]
baseline.attrs["provenance"] = "factor_baseline"
baseline.attrs["expression"] = "$close/Ref($close,10)-1"

print(f"baseline shape: {baseline.shape}")
print(baseline.head())

In [ ]:
# ── Run 10D experiment: compare true ranker vs historical momentum baseline ──
experiment = run_10d_experiment(
    config=config,
    candidates={
        "lgbm:daily_ranker": ranker_scores,
        "factor:historical_momentum_10d": baseline,
    },
    raw_returns=returns_test,
    output_dir=ROOT / "artifacts" / "evidence" / "notebook_10d_lab",
)

report = experiment["comparison_report"]
print(json.dumps(report["summary"], indent=2, default=str))
print(json.dumps(summarize_report_gates(report), indent=2, default=str))

# ── Contract: both exact candidate names must appear in the report ──
candidates = report.get("candidates", [])
candidate_names = {c.get("candidate_name", "") for c in candidates}
assert "lgbm:daily_ranker" in candidate_names, (
    "lgbm:daily_ranker must appear in comparison report candidate_name"
)
assert "factor:historical_momentum_10d" in candidate_names, (
    "factor:historical_momentum_10d must appear in comparison report candidate_name"
)
# Backward compat: name also embedded in strength_rationale
rationales = [c.get("strength_rationale", "") for c in candidates]
assert any("lgbm:daily_ranker:" in r for r in rationales), (
    "lgbm:daily_ranker must appear in comparison report strength_rationale"
)
assert any("factor:historical_momentum_10d:" in r for r in rationales), (
    "factor:historical_momentum_10d must appear in comparison report strength_rationale"
)
artifact_path = experiment.get("artifact_path")
assert artifact_path is not None, "artifact_path must be present"
assert Path(artifact_path).exists(), f"artifact file missing: {artifact_path}"
print(f"evidence artifact written → {artifact_path}")